# Raw Claim Retrieval Evaluation

Evaluate top-3 evidence retrieval against `FinalDataset/claims_merged.csv` using the raw claim text and claim image path only.

This notebook mirrors `database/retrieval_evaluation.ipynb`, but the retrieval input is not refined by an LLM:

- input claim text: `claim`
- input claim image: `image`
- Qdrant collection: `semantic`
- image vector: `image_vector_finetuned`
- reranker: on

Outputs are saved under `database/retrieval_eval_raw_outputs/`.


In [2]:
from __future__ import annotations

import hashlib
import json
import math
import re
import unicodedata
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseVector
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"database", "refined"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

QDRANT_URL = "http://localhost:6333"
GROUND_TRUTH_CSV = PROJECT_ROOT / "FinalDataset" / "claims_merged.csv"
OUTPUT_DIR = PROJECT_ROOT / "database" / "retrieval_eval_raw_outputs"
EXPERIMENT_OUTPUT_DIR = OUTPUT_DIR / "by_experiment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLLECTIONS = ["semantic"]
IMAGE_VECTOR_VARIANTS = {
    "clip_finetuned": "image_vector_finetuned",
}
RUN_RERANKER_VALUES = [True]
ROW_LIMIT = None  # Set to an integer for a quick smoke test.

TEXT_VECTOR = "text_vector"
SPARSE_VECTOR = "sparse"
BKVEC_MODEL = "bkai-foundation-models/vietnamese-bi-encoder"
IMG_MODEL = "sentence-transformers/clip-ViT-B-32"
IMG_MODEL_FINETUNED = PROJECT_ROOT / "models" / "clip-vit-b32-finetuned-final-final" / "best"
CROSS_ENCODER_MODEL = "namdp-ptit/ViRanker"
CROSS_ENCODER_MAX_LENGTH = 512

CANDIDATES_PER_QUERY = 10
MAX_TEXT_QUERIES = 6
MAX_VISUAL_QUERIES = 6
TOP_K_VALUES = [3, 10, 20]
FINAL_TOP_K = 3
RRF_K = 60
TOKEN_RE = re.compile(r"\w+", re.UNICODE)

client = QdrantClient(url=QDRANT_URL)
print("Project root:", PROJECT_ROOT)
print("Output dir:", OUTPUT_DIR)
print("Qdrant collections:", [c.name for c in client.get_collections().collections])


Project root: d:\FactCheckPipeline
Output dir: d:\FactCheckPipeline\database\retrieval_eval_raw_outputs
Qdrant collections: ['fixed_size', 'semantic']


## Load Raw Claims

The raw baseline uses `claim` as the text query and `image` as the claim image reference. The gold evidence columns stay in the same dataframe for evaluation.


In [3]:
gt = pd.read_csv(GROUND_TRUTH_CSV)
raw_claims = gt.copy()

required_cols = {"id", "claim", "image", "text_evidences", "text_evidences_url", "image_evidence_path"}
missing = sorted(required_cols - set(raw_claims.columns))
if missing:
    raise ValueError(f"Missing required columns in {GROUND_TRUTH_CSV}: {missing}")

print("raw_claims", raw_claims.shape)
display(raw_claims.head(3))


raw_claims (1293, 9)


,id,claim,image,text_evidences,text_evidences_url,image_evidences,image_evidence_path,reason,label
0,1,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng...,media/post_1_cmt_img_0.jpg,Công an tỉnh Thái Bình vừa triệt phá đường dây...,https://www.facebook.com/mps.gov/posts/pfbid02...,Hình ảnh bên phải hiển thị một nhóm khoảng 10-...,media/post_1_cmt_img_0.jpg,Claim này được đánh giá là supported vì văn bả...,supported
1,2,Các đối tượng cầm đầu đường dây lừa đảo đã thu...,media/post_1_cmt_img_0.jpg,Chúng thuê nhà tại Hà Nội và TP. Hồ Chí Minh l...,https://www.facebook.com/mps.gov/posts/pfbid02...,Hình ảnh bên trái hiển thị nhiều điện thoại di...,media/post_1_cmt_img_0.jpg,Claim này được đánh giá là supported vì văn bả...,supported
2,3,Thượng úy Nguyễn Đức Phước là Điều tra viên th...,media/post_1_cmt_img_0.jpg,Đại úy Nguyễn Đức Phước - Phó Đội trưởng Đội Đ...,https://www.facebook.com/mps.gov/posts/pfbid02...,Hình ảnh không cung cấp thông tin về vai trò c...,media/post_1_cmt_img_0.jpg,Claim này bị đánh giá là refuted vì văn bản cu...,refuted


## Matching Rules

Text evidence hit priority:

1. exact URL match
2. token coverage against each gold evidence fragment
3. token F1 against each gold evidence fragment

The coverage rule is important because retrieved chunks are often much longer than the gold evidence sentence. Pure F1 would unfairly penalize long but correct chunks.

In [4]:
def normalize_text(text: Any) -> str:
    text = str(text or "").lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text

def tokens(text: Any) -> list[str]:
    return TOKEN_RE.findall(normalize_text(text))

def token_scores(retrieved_text: Any, gold_text: Any) -> dict[str, float]:
    r = set(tokens(retrieved_text))
    g = set(tokens(gold_text))
    if not r or not g:
        return {"precision": 0.0, "coverage": 0.0, "f1": 0.0}
    inter = len(r & g)
    precision = inter / len(r)
    coverage = inter / len(g)
    f1 = 2 * precision * coverage / (precision + coverage) if precision + coverage else 0.0
    return {"precision": precision, "coverage": coverage, "f1": f1}

def split_gold_text_evidences(value: Any) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    return [part.strip() for part in str(value).split(" | ") if part.strip()]

def normalize_url(value: Any) -> str:
    text = str(value or "").strip().lower()
    return text.rstrip("/")

def normalize_path(value: Any) -> str:
    text = str(value or "").replace("\\", "/").strip().lower()
    while text.startswith("../"):
        text = text[3:]
    if text.startswith("finaldataset/"):
        text = text[len("finaldataset/"):]
    return text

def text_evidence_match(payload: dict[str, Any], gold_row: pd.Series) -> dict[str, Any]:
    retrieved_url = normalize_url(payload.get("url", ""))
    gold_url = normalize_url(gold_row.get("text_evidences_url", ""))
    if retrieved_url and gold_url and retrieved_url == gold_url:
        return {"hit": True, "method": "url", "coverage": 1.0, "f1": 1.0}

    retrieved_text = " ".join(str(payload.get(k, "")) for k in ["title", "text", "source", "url"])
    best = {"hit": False, "method": "none", "coverage": 0.0, "f1": 0.0}
    for gold_text in split_gold_text_evidences(gold_row.get("text_evidences", "")):
        scores = token_scores(retrieved_text, gold_text)
        if scores["coverage"] > best["coverage"] or scores["f1"] > best["f1"]:
            best = {"hit": False, "method": "text_overlap", **scores}
    best["hit"] = best["coverage"] >= 0.60 or best["f1"] >= 0.45
    return best

def image_evidence_match(payload: dict[str, Any], gold_row: pd.Series) -> dict[str, Any]:
    retrieved = normalize_path(payload.get("image_path", ""))
    gold = normalize_path(gold_row.get("image_evidence_path", ""))
    hit = bool(retrieved and gold and retrieved == gold)
    return {"hit": hit, "method": "path" if hit else "none"}


## Query Pack And Embeddings

For the raw baseline, the same raw claim text drives all retrieval lanes:

- dense text retrieval with the BKAI Vietnamese bi-encoder
- sparse text retrieval with the hashed sparse vector used by the original notebook
- image retrieval by encoding the raw claim text with the fine-tuned CLIP text encoder and querying `image_vector_finetuned`


In [5]:
def safe_json(value: Any, default: Any):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return default
    if isinstance(value, (list, dict)):
        return value
    try:
        return json.loads(str(value))
    except Exception:
        return default

def dedupe(items: list[Any], max_items: int | None = None) -> list[str]:
    out, seen = [], set()
    for item in items:
        text = str(item or "").strip()
        key = normalize_text(text)
        if text and key not in seen:
            seen.add(key)
            out.append(text)
    return out[:max_items] if max_items else out

def build_query_pack(row: pd.Series) -> dict[str, Any]:
    claim_text = str(row.get("claim", "")).strip()
    text_queries = dedupe([claim_text], MAX_TEXT_QUERIES)
    visual_queries = dedupe([claim_text], MAX_VISUAL_QUERIES)
    return {
        "text_queries": text_queries,
        "keyword_query": claim_text,
        "visual_queries": visual_queries,
        "retrieval_focus": {"text": True, "image": True, "cross_modal": True},
    }

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device", device)
bkai_model = SentenceTransformer(BKVEC_MODEL, device=device)
clip_model = SentenceTransformer(IMG_MODEL, device=device)
clip_finetuned_model = SentenceTransformer(str(IMG_MODEL_FINETUNED), device=device)

def embed_text_bkai(texts: list[str]) -> list[list[float]]:
    return bkai_model.encode(texts, batch_size=32, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False).tolist()

def embed_text_clip(texts: list[str], image_variant: str) -> list[list[float]]:
    model = clip_finetuned_model if image_variant == "clip_finetuned" else clip_model
    return model.encode(texts, batch_size=16, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False).tolist()

def sparse_vector(text: str) -> SparseVector:
    counts = {}
    for token in TOKEN_RE.findall(str(text or "").lower()):
        digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
        idx = int.from_bytes(digest, "big") % 2_147_483_647
        counts[idx] = counts.get(idx, 0) + 1
    indices = sorted(counts)
    values = [1.0 + math.log(counts[idx]) for idx in indices]
    norm = math.sqrt(sum(v * v for v in values)) or 1.0
    return SparseVector(indices=indices, values=[v / norm for v in values])


device cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## Retrieval, Fusion, Optional Reranker

In [6]:
def modality_filter(modality: str) -> models.Filter:
    return models.Filter(must=[models.FieldCondition(key="modality", match=models.MatchValue(value=modality))])

def query_qdrant(collection: str, query: Any, vector_name: str, modality: str, limit: int):
    return client.query_points(
        collection_name=collection,
        query=query,
        using=vector_name,
        query_filter=modality_filter(modality),
        limit=limit,
        with_payload=True,
        with_vectors=False,
    ).points

def generate_candidates(row: pd.Series, collection: str, image_variant: str) -> dict[str, list[dict[str, Any]]]:
    pack = build_query_pack(row)
    branches = {}

    text_hits = []
    if pack["text_queries"]:
        for q, v in zip(pack["text_queries"], embed_text_bkai(pack["text_queries"])):
            text_hits.extend({"point": p, "query": q} for p in query_qdrant(collection, v, TEXT_VECTOR, "text", CANDIDATES_PER_QUERY))
    branches["text_dense"] = text_hits

    sparse_hits = []
    if pack["keyword_query"].strip():
        sparse_hits = [{"point": p, "query": pack["keyword_query"]} for p in query_qdrant(collection, sparse_vector(pack["keyword_query"]), SPARSE_VECTOR, "text", CANDIDATES_PER_QUERY * 2)]
    branches["text_sparse"] = sparse_hits

    image_hits = []
    vector_name = IMAGE_VECTOR_VARIANTS[image_variant]
    if pack["visual_queries"]:
        for q, v in zip(pack["visual_queries"], embed_text_clip(pack["visual_queries"], image_variant)):
            image_hits.extend({"point": p, "query": q} for p in query_qdrant(collection, v, vector_name, "image", CANDIDATES_PER_QUERY))
    branches[f"image_{image_variant}"] = image_hits
    return branches

# Equal weights keep original CLIP vs fine-tuned CLIP comparable.
# Text and image are evaluated in separate lanes, so the image lane is not pushed down by the text reranker.
BRANCH_WEIGHTS = {"text_dense": 1.00, "text_sparse": 1.00, "image_clip": 1.00, "image_clip_finetuned": 1.00}

def weighted_rrf(branches: dict[str, list[dict[str, Any]]]) -> list[dict[str, Any]]:
    fused = {}
    for branch, records in branches.items():
        seen = set()
        for rank, record in enumerate(records, start=1):
            p = record["point"]
            key = str(p.id)
            if key in seen:
                continue
            seen.add(key)
            item = fused.setdefault(key, {"point_id": key, "payload": p.payload or {}, "rrf_score": 0.0, "branches": [], "best_qdrant_score": float(p.score)})
            item["rrf_score"] += BRANCH_WEIGHTS.get(branch, 1.0) / (RRF_K + rank)
            item["best_qdrant_score"] = max(item["best_qdrant_score"], float(p.score))
            item["branches"].append({"branch": branch, "rank": rank, "score": float(p.score), "query": record.get("query", "")})
    return sorted(fused.values(), key=lambda x: x["rrf_score"], reverse=True)

cross_encoder = None
def get_cross_encoder():
    global cross_encoder
    if cross_encoder is None:
        from sentence_transformers import CrossEncoder
        cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, device=device, max_length=CROSS_ENCODER_MAX_LENGTH)
    return cross_encoder

def rerank_items(items: list[dict[str, Any]], row: pd.Series, use_reranker: bool) -> list[dict[str, Any]]:
    query = str(row.get("claim") or "")
    has_text_candidate = any((item.get("payload") or {}).get("modality") == "text" for item in items)
    ce = get_cross_encoder() if use_reranker and has_text_candidate else None
    for item in items:
        payload = item["payload"]
        branch_bonus = min(len({b["branch"] for b in item["branches"]}) * 0.025, 0.10)
        reranker_boost = 0.0
        if ce is not None and payload.get("modality") == "text":
            passage = " ".join(str(payload.get(k, "")) for k in ["title", "text", "source", "date"])
            if passage.strip():
                raw = float(ce.predict([(query, passage[:3000])])[0])
                reranker_boost = (1.0 / (1.0 + math.exp(-raw))) * 0.25
        item["reranker_boost"] = reranker_boost
        item["final_score"] = item["rrf_score"] + branch_bonus + reranker_boost
    return sorted(items, key=lambda x: x["final_score"], reverse=True)

def retrieve_lanes(row: pd.Series, collection: str, image_variant: str, use_reranker: bool) -> dict[str, list[dict[str, Any]]]:
    branches = generate_candidates(row, collection, image_variant)
    text_branches = {name: records for name, records in branches.items() if name.startswith("text_")}
    image_branches = {name: records for name, records in branches.items() if name.startswith("image_")}
    text_ranked = rerank_items(weighted_rrf(text_branches), row, use_reranker)
    image_ranked = rerank_items(weighted_rrf(image_branches), row, False)
    return {"text": text_ranked, "image": image_ranked}


## Metrics

In [7]:
def source_url_hit(payload: dict[str, Any], gold_row: pd.Series) -> bool:
    retrieved_url = normalize_url(payload.get("url", ""))
    gold_url = normalize_url(gold_row.get("text_evidences_url", ""))
    return bool(retrieved_url and gold_url and retrieved_url == gold_url)

def serialize_top_url_items(items: list[dict[str, Any]], limit: int = 3) -> str:
    records = []
    for rank, item in enumerate(items[:limit], start=1):
        payload = item["payload"]
        records.append({
            "rank": rank,
            "point_id": item["point_id"],
            "url": payload.get("url", ""),
            "title": payload.get("title", ""),
            "image_path": payload.get("image_path", ""),
            "score": item.get("final_score", 0.0),
        })
    return json.dumps(records, ensure_ascii=False)

def serialize_top_urls(items: list[dict[str, Any]], limit: int = 3) -> str:
    return json.dumps([item["payload"].get("url", "") for item in items[:limit]], ensure_ascii=False)

def build_result_row(item: dict[str, Any], rank: int, lane: str, gold_row: pd.Series) -> dict[str, Any]:
    payload = item["payload"]
    text_match = text_evidence_match(payload, gold_row) if payload.get("modality") == "text" else {"hit": False, "method": "none", "coverage": 0.0, "f1": 0.0}
    image_match = image_evidence_match(payload, gold_row) if payload.get("modality") == "image" else {"hit": False, "method": "none"}
    evidence_hit = bool(text_match["hit"] or image_match["hit"])
    return {
        "lane": lane,
        "rank": rank,
        "point_id": item["point_id"],
        "modality": payload.get("modality", ""),
        "final_score": item.get("final_score", 0.0),
        "rrf_score": item.get("rrf_score", 0.0),
        "reranker_boost": item.get("reranker_boost", 0.0),
        "best_qdrant_score": item.get("best_qdrant_score", 0.0),
        "branches": json.dumps(item.get("branches", []), ensure_ascii=False),
        "hit": evidence_hit,
        "source_hit": source_url_hit(payload, gold_row),
        "text_hit": bool(text_match["hit"]),
        "image_hit": bool(image_match["hit"]),
        "text_match_method": text_match.get("method", "none"),
        "text_coverage": text_match.get("coverage", 0.0),
        "text_f1": text_match.get("f1", 0.0),
        "title": payload.get("title", ""),
        "url": payload.get("url", ""),
        "image_path": payload.get("image_path", ""),
        "text": str(payload.get("text", ""))[:1000],
    }

def evaluate_ranked_lanes(ranked_lanes: dict[str, list[dict[str, Any]]], gold_row: pd.Series) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    text_items = ranked_lanes.get("text", [])
    image_items = ranked_lanes.get("image", [])
    text_rows = [build_result_row(item, rank, "text", gold_row) for rank, item in enumerate(text_items[:max(TOP_K_VALUES)], start=1)]
    image_rows = [build_result_row(item, rank, "image", gold_row) for rank, item in enumerate(image_items[:max(TOP_K_VALUES)], start=1)]
    rows = text_rows + image_rows

    metrics = {
        "top3_text_urls": serialize_top_urls(text_items, 3),
        "top3_image_urls": serialize_top_urls(image_items, 3),
        "top3_text_items": serialize_top_url_items(text_items, 3),
        "top3_image_items": serialize_top_url_items(image_items, 3),
    }
    for k in TOP_K_VALUES:
        text_top = text_rows[:k]
        image_top = image_rows[:k]
        metrics[f"text_source_hit_at_{k}"] = int(any(r["source_hit"] for r in text_top))
        metrics[f"image_source_hit_at_{k}"] = int(any(r["source_hit"] for r in image_top))
        metrics[f"source_hit_at_{k}"] = int(metrics[f"text_source_hit_at_{k}"] or metrics[f"image_source_hit_at_{k}"])
        metrics[f"text_evidence_hit_at_{k}"] = int(any(r["text_hit"] for r in text_top))
        metrics[f"image_exact_hit_at_{k}"] = int(any(r["image_hit"] for r in image_top))
        metrics[f"evidence_hit_at_{k}"] = int(metrics[f"text_evidence_hit_at_{k}"] or metrics[f"image_exact_hit_at_{k}"])

    first_text_source_rank = next((r["rank"] for r in text_rows if r["source_hit"]), None)
    first_image_source_rank = next((r["rank"] for r in image_rows if r["source_hit"]), None)
    first_text_evidence_rank = next((r["rank"] for r in text_rows if r["text_hit"]), None)
    first_image_exact_rank = next((r["rank"] for r in image_rows if r["image_hit"]), None)

    metrics["first_text_source_hit_rank"] = first_text_source_rank or 0
    metrics["first_image_source_hit_rank"] = first_image_source_rank or 0
    metrics["first_text_evidence_hit_rank"] = first_text_evidence_rank or 0
    metrics["first_image_exact_hit_rank"] = first_image_exact_rank or 0
    metrics["text_source_mrr_at_3"] = 1 / first_text_source_rank if first_text_source_rank and first_text_source_rank <= 3 else 0.0
    metrics["image_source_mrr_at_3"] = 1 / first_image_source_rank if first_image_source_rank and first_image_source_rank <= 3 else 0.0
    metrics["source_mrr_at_3"] = max(metrics["text_source_mrr_at_3"], metrics["image_source_mrr_at_3"])
    metrics["text_evidence_mrr_at_3"] = 1 / first_text_evidence_rank if first_text_evidence_rank and first_text_evidence_rank <= 3 else 0.0
    metrics["image_exact_mrr_at_3"] = 1 / first_image_exact_rank if first_image_exact_rank and first_image_exact_rank <= 3 else 0.0
    metrics["evidence_mrr_at_3"] = max(metrics["text_evidence_mrr_at_3"], metrics["image_exact_mrr_at_3"])
    return rows, metrics

def summarize_metrics(per_claim_metrics: list[dict[str, Any]]) -> dict[str, Any]:
    frame = pd.DataFrame(per_claim_metrics)
    metric_cols = [
        c for c in frame.columns
        if any(c.endswith(f"_at_{k}") for k in TOP_K_VALUES)
        or c.endswith("_mrr_at_3")
    ]
    return {col: float(frame[col].mean()) for col in metric_cols}


## Run Raw-Claim Experiment


In [8]:
all_detail_rows = []
all_claim_metric_rows = []
all_top3_url_rows = []
summary_rows = []

query_source = "raw_claim"
eval_frame = raw_claims.copy()
if ROW_LIMIT is not None:
    eval_frame = eval_frame.head(ROW_LIMIT).copy()

print(f"{query_source}: evaluating {len(eval_frame)} rows")
for collection in COLLECTIONS:
    for image_variant in IMAGE_VECTOR_VARIANTS:
        for use_reranker in RUN_RERANKER_VALUES:
            experiment_id = f"{query_source}__{collection}__{image_variant}__reranker_{int(use_reranker)}"
            print("Running", experiment_id)
            detail_rows = []
            claim_metric_rows = []
            top3_url_rows = []
            for _, row in tqdm(eval_frame.iterrows(), total=len(eval_frame), desc=experiment_id):
                ranked_lanes = retrieve_lanes(row, collection, image_variant, use_reranker)
                evidence_rows, metrics = evaluate_ranked_lanes(ranked_lanes, row)
                base = {
                    "experiment_id": experiment_id,
                    "refiner": query_source,
                    "collection": collection,
                    "image_variant": image_variant,
                    "use_reranker": use_reranker,
                    "id": row["id"],
                    "claim": row["claim"],
                    "label": row.get("label", ""),
                    "gold_text_url": row.get("text_evidences_url", ""),
                    "gold_image_path": row.get("image_evidence_path", ""),
                }
                for ev in evidence_rows:
                    detail_rows.append({**base, **ev})
                claim_metric_rows.append({**base, **metrics})
                top3_url_rows.append({
                    **base,
                    "top3_text_urls": metrics["top3_text_urls"],
                    "top3_image_urls": metrics["top3_image_urls"],
                    "top3_text_items": metrics["top3_text_items"],
                    "top3_image_items": metrics["top3_image_items"],
                    "text_source_hit_at_3": metrics["text_source_hit_at_3"],
                    "image_source_hit_at_3": metrics["image_source_hit_at_3"],
                    "source_hit_at_3": metrics["source_hit_at_3"],
                })

            detail_df = pd.DataFrame(detail_rows)
            claim_metrics_df = pd.DataFrame(claim_metric_rows)
            top3_urls_df = pd.DataFrame(top3_url_rows)
            summary = summarize_metrics(claim_metric_rows)
            summary_rows.append({
                "experiment_id": experiment_id,
                "refiner": query_source,
                "collection": collection,
                "image_variant": image_variant,
                "use_reranker": use_reranker,
                "num_claims": len(claim_metric_rows),
                **summary,
            })

            experiment_dir = EXPERIMENT_OUTPUT_DIR / experiment_id
            experiment_dir.mkdir(parents=True, exist_ok=True)
            detail_path = experiment_dir / "retrieval_details.csv"
            claim_metrics_path = experiment_dir / "claim_metrics.csv"
            top3_urls_path = experiment_dir / "top3_urls.csv"
            detail_df.to_csv(detail_path, index=False, encoding="utf-8-sig")
            claim_metrics_df.to_csv(claim_metrics_path, index=False, encoding="utf-8-sig")
            top3_urls_df.to_csv(top3_urls_path, index=False, encoding="utf-8-sig")
            all_detail_rows.extend(detail_rows)
            all_claim_metric_rows.extend(claim_metric_rows)
            all_top3_url_rows.extend(top3_url_rows)

summary_df = pd.DataFrame(summary_rows).sort_values(["source_hit_at_3", "source_mrr_at_3", "text_source_hit_at_3"], ascending=False)
all_details_df = pd.DataFrame(all_detail_rows)
all_claim_metrics_df = pd.DataFrame(all_claim_metric_rows)
all_top3_urls_df = pd.DataFrame(all_top3_url_rows)

summary_df.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False, encoding="utf-8-sig")
all_details_df.to_csv(OUTPUT_DIR / "retrieval_results_long.csv", index=False, encoding="utf-8-sig")
all_claim_metrics_df.to_csv(OUTPUT_DIR / "claim_metrics_long.csv", index=False, encoding="utf-8-sig")
all_top3_urls_df.to_csv(OUTPUT_DIR / "top3_urls_long.csv", index=False, encoding="utf-8-sig")

display(summary_df)
print("Saved outputs to", OUTPUT_DIR)


raw_claim: evaluating 1293 rows
Running raw_claim__semantic__clip_finetuned__reranker_1


raw_claim__semantic__clip_finetuned__reranker_1:   0%|          | 0/1293 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

,experiment_id,refiner,collection,image_variant,use_reranker,num_claims,text_source_hit_at_3,image_source_hit_at_3,source_hit_at_3,text_evidence_hit_at_3,...,source_hit_at_20,text_evidence_hit_at_20,image_exact_hit_at_20,evidence_hit_at_20,text_source_mrr_at_3,image_source_mrr_at_3,source_mrr_at_3,text_evidence_mrr_at_3,image_exact_mrr_at_3,evidence_mrr_at_3
0,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True,1293,0.825213,0.000773,0.825986,0.924207,...,0.898685,0.958237,0.0,0.958237,0.762568,0.000258,0.762825,0.886182,0.0,0.886182


Saved outputs to d:\FactCheckPipeline\database\retrieval_eval_raw_outputs


## Best Raw-Claim Result

Cell này đọc `metrics_summary.csv` từ raw-claim run và hiển thị summary theo cùng schema với notebook evaluation hiện tại.


In [9]:
summary_path = OUTPUT_DIR / "metrics_summary.csv"
if not summary_path.exists():
    raise FileNotFoundError(f"Run the evaluation cell first. Missing: {summary_path}")

metrics_summary = pd.read_csv(summary_path)
id_cols = ["experiment_id", "refiner", "collection", "image_variant", "use_reranker", "num_claims"]
metric_cols = [
    c for c in metrics_summary.columns
    if c not in id_cols and pd.api.types.is_numeric_dtype(metrics_summary[c])
]

preferred_sort_cols = [
    c for c in [
        "source_hit_at_3",
        "source_mrr_at_3",
        "text_source_hit_at_3",
        "image_source_hit_at_3",
        "evidence_hit_at_3",
    ]
    if c in metrics_summary.columns
]
sorted_summary = metrics_summary.sort_values(preferred_sort_cols, ascending=False) if preferred_sort_cols else metrics_summary

best_rows = []
for metric in metric_cols:
    best_value = metrics_summary[metric].max()
    winners = metrics_summary[metrics_summary[metric] == best_value].copy()
    for _, row in winners.iterrows():
        best_rows.append({
            "metric": metric,
            "best_value": best_value,
            "experiment_id": row["experiment_id"],
            "refiner": row["refiner"],
            "collection": row["collection"],
            "image_variant": row["image_variant"],
            "use_reranker": row["use_reranker"],
        })

best_by_metric = pd.DataFrame(best_rows).sort_values(["metric", "experiment_id"])

print("Metrics summary sorted by main source metrics")
display(sorted_summary)

print("Best combination(s) for each metric")
display(best_by_metric)

best_by_metric.to_csv(OUTPUT_DIR / "best_combinations_by_metric.csv", index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_DIR / "best_combinations_by_metric.csv")


Metrics summary sorted by main source metrics


,experiment_id,refiner,collection,image_variant,use_reranker,num_claims,text_source_hit_at_3,image_source_hit_at_3,source_hit_at_3,text_evidence_hit_at_3,...,source_hit_at_20,text_evidence_hit_at_20,image_exact_hit_at_20,evidence_hit_at_20,text_source_mrr_at_3,image_source_mrr_at_3,source_mrr_at_3,text_evidence_mrr_at_3,image_exact_mrr_at_3,evidence_mrr_at_3
0,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True,1293,0.825213,0.000773,0.825986,0.924207,...,0.898685,0.958237,0.0,0.958237,0.762568,0.000258,0.762825,0.886182,0.0,0.886182


Best combination(s) for each metric


,metric,best_value,experiment_id,refiner,collection,image_variant,use_reranker
11,evidence_hit_at_10,0.951276,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True
17,evidence_hit_at_20,0.958237,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True
5,evidence_hit_at_3,0.924207,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True
23,evidence_mrr_at_3,0.886182,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True
10,image_exact_hit_at_10,0.000000,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True
16,image_exact_hit_at_20,0.000000,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True
4,image_exact_hit_at_3,0.000000,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True
22,image_exact_mrr_at_3,0.000000,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True
7,image_source_hit_at_10,0.002320,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True
13,image_source_hit_at_20,0.002320,raw_claim__semantic__clip_finetuned__reranker_1,raw_claim,semantic,clip_finetuned,True


Saved: d:\FactCheckPipeline\database\retrieval_eval_raw_outputs\best_combinations_by_metric.csv
